# Credit Scoring — Fase 2: Preprocesamiento y Feature Engineering

Construimos el pipeline de transformación completo: imputación, capping de outliers, nuevas features, WoE encoding, y split train/val/test.  
El resultado son los datasets limpios en `data/processed/` y el `WoEEncoder` serializado.

In [ ]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split

sys.path.append('..')
from src.features.woe_encoder import WoEEncoder

DATA_RAW = Path('../data/raw')
DATA_PROC = Path('../data/processed')
MODELS = Path('../models/saved')
FIGURES = Path('../reports/figures')
DATA_PROC.mkdir(exist_ok=True)
MODELS.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

## 1. Carga y renombrado de columnas

In [ ]:
df = pd.read_csv(DATA_RAW / 'cs-training.csv', index_col=0)

df = df.rename(columns={
    'SeriousDlqin2yrs': 'target',
    'RevolvingUtilizationOfUnsecuredLines': 'revolving_util',
    'NumberOfTime30-59DaysPastDueNotWorse': 'late_30_59',
    'NumberOfTime60-89DaysPastDueNotWorse': 'late_60_89',
    'NumberOfTimes90DaysLate': 'late_90plus',
    'DebtRatio': 'debt_ratio',
    'MonthlyIncome': 'monthly_income',
    'NumberOfOpenCreditLinesAndLoans': 'open_credit_lines',
    'NumberRealEstateLoansOrLines': 'real_estate_loans',
    'NumberOfDependents': 'n_dependents',
})

print(f'Shape: {df.shape}')
print(f'Nulos:\n{df.isnull().sum()[df.isnull().sum() > 0]}')

## 2. Imputación de valores nulos

- `monthly_income`: mediana por grupo de edad (correlación ingreso-edad más precisa que mediana global)
- `n_dependents`: moda global (variable discreta con pocos valores)

In [ ]:
# Grupos de edad para imputación estratificada
df['age_group_tmp'] = pd.cut(df['age'], bins=[0, 30, 45, 60, 120],
                              labels=['18-30', '31-45', '46-60', '60+'])

# Mediana de monthly_income por grupo de edad
income_medians = df.groupby('age_group_tmp', observed=True)['monthly_income'].median()
print('Mediana de ingreso por grupo de edad:')
print(income_medians.to_string())

df['monthly_income'] = df.apply(
    lambda row: income_medians[row['age_group_tmp']]
    if pd.isnull(row['monthly_income']) else row['monthly_income'],
    axis=1
)

# Moda de n_dependents
mode_dependents = df['n_dependents'].mode()[0]
df['n_dependents'] = df['n_dependents'].fillna(mode_dependents)

df.drop(columns=['age_group_tmp'], inplace=True)

print(f'\nNulos restantes: {df.isnull().sum().sum()}')

## 3. Tratamiento de outliers con capping (percentiles 1% y 99%)

Capping preserva todas las filas — solo trunca los valores extremos que distorsionarían el modelo.

In [ ]:
features_to_cap = [
    'revolving_util', 'debt_ratio', 'monthly_income',
    'late_30_59', 'late_60_89', 'late_90plus',
    'open_credit_lines', 'real_estate_loans', 'n_dependents'
]

cap_bounds = {}
for col in features_to_cap:
    low, high = df[col].quantile([0.01, 0.99])
    cap_bounds[col] = (low, high)
    df[col] = df[col].clip(lower=low, upper=high)

cap_df = pd.DataFrame(cap_bounds, index=['p01', 'p99']).T
print('Límites de capping:')
display(cap_df.round(3))

# Guardar bounds para usarlos en transform de producción
joblib.dump(cap_bounds, MODELS / 'cap_bounds.joblib')

## 4. Feature Engineering

Creamos features derivadas que capturan relaciones económicas relevantes para credit scoring.

In [ ]:
# Ratio deuda-ingreso efectivo (DebtRatio es ratio mensual, monthly_income es el denominador real)
df['debt_to_income'] = df['debt_ratio'] * df['monthly_income']

# Total de atrasos de cualquier tipo
df['total_late_payments'] = df['late_30_59'] + df['late_60_89'] + df['late_90plus']

# Ingreso por dependiente (proxy de presión financiera)
df['income_per_dependent'] = df['monthly_income'] / (df['n_dependents'] + 1)

# Grupos de utilización de crédito
df['util_bucket'] = pd.cut(
    df['revolving_util'],
    bins=[-np.inf, 0.3, 0.6, 0.8, np.inf],
    labels=['low', 'medium', 'high', 'very_high']
).astype(str)

# Grupos etarios
df['age_group'] = pd.cut(
    df['age'],
    bins=[0, 30, 45, 60, 120],
    labels=['18-30', '31-45', '46-60', '60+']
).astype(str)

print('Features creadas:')
new_features = ['debt_to_income', 'total_late_payments', 'income_per_dependent', 'util_bucket', 'age_group']
display(df[new_features].describe().T.round(2))

## 5. Split train / validation / test (60/20/20)

Estratificado por target para mantener la misma proporción de defaults en cada split.

In [ ]:
# Separar features categóricas — no entran al WoE encoder (se encodean separado)
cat_features = ['util_bucket', 'age_group']
num_features = [
    'revolving_util', 'age', 'late_30_59', 'debt_ratio', 'monthly_income',
    'open_credit_lines', 'late_90plus', 'real_estate_loans', 'late_60_89',
    'n_dependents', 'debt_to_income', 'total_late_payments', 'income_per_dependent'
]

X = df[num_features + cat_features]
y = df['target']

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.25, stratify=y_train_full, random_state=42
)  # 0.25 * 0.8 = 0.2 del total

print(f'Train:      {X_train.shape}  | default rate: {y_train.mean():.2%}')
print(f'Validation: {X_val.shape}   | default rate: {y_val.mean():.2%}')
print(f'Test:       {X_test.shape}   | default rate: {y_test.mean():.2%}')

## 6. WoE Encoding

Ajustamos el encoder **solo sobre train** para evitar data leakage. Las variables categóricas (`util_bucket`, `age_group`) también se encodean por WoE usando `pd.Categorical`.

In [ ]:
# Encoder para variables numéricas
woe_encoder = WoEEncoder(n_bins=10)
woe_encoder.fit(X_train[num_features], y_train)

print('IV por variable:')
display(woe_encoder.get_iv_table())

In [ ]:
# Visualización del IV
iv_table = woe_encoder.get_iv_table()

colors = iv_table['IV'].map(
    lambda v: 'tomato' if v > 0.5 else 'steelblue' if v >= 0.1 else
              'lightsteelblue' if v >= 0.02 else 'lightgray'
)

fig, ax = plt.subplots(figsize=(9, 5))
iv_table['IV'].plot(kind='barh', ax=ax, color=colors.values, edgecolor='white')
ax.axvline(0.1, color='orange', linestyle='--', linewidth=1, label='IV=0.1')
ax.axvline(0.3, color='green', linestyle='--', linewidth=1, label='IV=0.3')
ax.set_title('Information Value por variable (calculado sobre train)')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES / '09_iv_woe_encoder.png', bbox_inches='tight')
plt.show()

## 7. Generar datasets finales y serializar

Guardamos las versiones procesadas (con WoE y sin WoE) para usar en los notebooks de modelado.

In [ ]:
# Transformar splits con WoE
X_train_woe = woe_encoder.transform(X_train[num_features])
X_val_woe   = woe_encoder.transform(X_val[num_features])
X_test_woe  = woe_encoder.transform(X_test[num_features])

# Datasets raw (para modelos de árbol que no necesitan WoE)
train_raw = X_train[num_features].assign(target=y_train.values)
val_raw   = X_val[num_features].assign(target=y_val.values)
test_raw  = X_test[num_features].assign(target=y_test.values)

# Datasets con WoE (para scorecard / logistic regression)
train_woe = X_train_woe.assign(target=y_train.values)
val_woe   = X_val_woe.assign(target=y_val.values)
test_woe  = X_test_woe.assign(target=y_test.values)

# Guardar
train_raw.to_csv(DATA_PROC / 'train_raw.csv', index=False)
val_raw.to_csv(DATA_PROC / 'val_raw.csv', index=False)
test_raw.to_csv(DATA_PROC / 'test_raw.csv', index=False)

train_woe.to_csv(DATA_PROC / 'train_woe.csv', index=False)
val_woe.to_csv(DATA_PROC / 'val_woe.csv', index=False)
test_woe.to_csv(DATA_PROC / 'test_woe.csv', index=False)

# Serializar encoder
joblib.dump(woe_encoder, MODELS / 'woe_encoder.joblib')

print('Archivos guardados en data/processed/ y models/saved/')
print(f'\nTrain WoE shape: {train_woe.shape}')
display(train_woe.head(3))

## Resumen Fase 2

**Transformaciones aplicadas:**
- Imputación de `monthly_income` por mediana de grupo de edad, `n_dependents` por moda
- Capping p1/p99 en 9 variables numéricas → bounds guardados en `models/saved/cap_bounds.joblib`
- 5 nuevas features: `debt_to_income`, `total_late_payments`, `income_per_dependent`, `util_bucket`, `age_group`
- Split 60/20/20 estratificado
- `WoEEncoder` ajustado en train → serializado en `models/saved/woe_encoder.joblib`

**Datasets en `data/processed/`:**
- `train/val/test_raw.csv` — para modelos de árbol (XGBoost, LightGBM, etc.)
- `train/val/test_woe.csv` — para Logistic Regression y scorecard

**Próximo paso:** `03_modeling_baseline.ipynb`